# Guía Práctica 04: Gobernanza, Ética y Anonimización de Datos
## Proyecto: HRTech People Analytics - Equipo 8

Este notebook implementa las técnicas de seudonimización y anonimización sobre el dataset consolidado de personal, y genera automáticamente la bitácora de trabajo en formato `.md` al finalizar el pipeline.

### 1. Carga de Librerías y Dataset Consolidado

In [4]:
import hashlib
from pathlib import Path
import pandas as pd
import numpy as np

# Configuración de rutas
BASE_DIR = Path("../")
ruta_consolidado = BASE_DIR / "datos" / "procesados" / "dataset_consolidado.csv"

# Carga de datos o fallback a mock de pruebas
try:
    df = pd.read_csv(ruta_consolidado)
    print(f"Dataset cargado exitosamente desde: {ruta_consolidado}")
except Exception:
    print(f"Nota: Carga de datos de demostración para ejecuciones locales sin archivo físico.")
    data_mock = {
        'empleado_id': ['EMP-001', 'EMP-002', 'EMP-003', 'EMP-004'],
        'nombre_completo': ['Carlos Pérez', 'Ana Gómez', 'Luis Martínez', 'María Rodríguez'],
        'departamento': ['TI', 'RRHH', 'Finanzas', 'TI'],
        'puesto': ['Desarrollador', 'Analista', 'Contador', 'Líder Técnico'],
        'salario_mensual': [1200.00, 1450.00, 2100.00, 2800.00],
        'edad': [24, 31, 42, 53],
        'genero': ['M', 'F', 'M', 'F'],
        'latitud': [13.67, 13.68, 13.69, 13.67],
        'longitud': [-89.28, -89.29, -89.27, -89.28],
        'direccion_referencia': ['Av. Principal #1', 'Calle A #2', 'Zona 3', 'Colonia B']
    }
    df = pd.DataFrame(data_mock)

df.head()

Dataset cargado exitosamente desde: ..\datos\procesados\dataset_consolidado.csv


,registro_id,empleado_id,periodo,fecha_evaluacion,productividad_pct,calidad_trabajo_pct,cumplimiento_objetivos_pct,asistencia_pct,horas_capacitacion,horas_extra,...,respuestas_intencion_de_permanecer,ubicacion_laboral_sede_codigo,ubicacion_laboral_pais,ubicacion_laboral_departamento,ubicacion_laboral_municipio,ubicacion_laboral_distrito,ubicacion_laboral_direccion_referencia,ubicacion_laboral_latitud,ubicacion_laboral_longitud,ubicacion_laboral_zona
0,1,EMP001,2025-Q3,2025-09-30,76.2,72.1,68.6,97.6,12,9,...,3,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro
1,2,EMP001,2025-Q4,2025-12-31,56.2,76.7,71.5,94.1,11,12,...,3,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro
2,3,EMP001,2026-Q1,2026-03-31,75.1,67.9,76.0,92.8,10,19,...,3,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro
3,4,EMP001,2026-Q2,2026-06-30,71.6,79.8,78.1,93.3,9,8,...,3,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro
4,5,EMP002,2025-Q3,2025-09-30,76.5,80.2,86.3,98.7,4,5,...,5,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.671427,-89.204773,Centro


### 2. Seudonimización y Anonimización

In [5]:
SALT = "HRTech_2026_Secure_Key"

def hashear_valor(valor):
    if pd.isna(valor):
        return np.nan
    cadena = str(valor) + SALT
    return hashlib.sha256(cadena.encode('utf-8')).hexdigest()[:16]

# Seudonimización de IDs y eliminación de PII directo
df['empleado_id_hash'] = df['empleado_id'].apply(hashear_valor)
df_anonimizado = df.drop(columns=['nombre_completo', 'empleado_id'])

# Generalización (Banding)
bins_salario = [0, 1000, 1500, 2000, 2500, 5000]
labels_salario = ['<$1,000', '$1,000-$1,499', '$1,500-$1,999', '$2,000-$2,499', '>=$2,500']
df_anonimizado['rango_salarial'] = pd.cut(df['salario_mensual'], bins=bins_salario, labels=labels_salario, right=False)
df_anonimizado = df_anonimizado.drop(columns=['salario_mensual'])

bins_edad = [18, 25, 35, 45, 55, 100]
labels_edad = ['18-24', '25-34', '35-44', '45-54', '55+']
df_anonimizado['rango_edad'] = pd.cut(df['edad'], bins=bins_edad, labels=labels_edad, right=False)
df_anonimizado = df_anonimizado.drop(columns=['edad'])

# Supresión de geodatos
col_geo = ['latitud', 'longitud', 'direccion_referencia']
df_anonimizado = df_anonimizado.drop(columns=[c for c in col_geo if c in df_anonimizado.columns])

# Exportar dataset resultantes
df_anonimizado.to_csv("../datos/procesados/dataset_anonimizado.csv", index=False, encoding='utf-8')
print("Proceso de anonimización completado exitosamente.")

Proceso de anonimización completado exitosamente.


### 3. Generación Automática de la Bitácora (`bitacora_guia04.md`)
Ejecuta esta celda para exportar o regenerar el archivo `bitacora_guia04.md` en el directorio de trabajo.

In [6]:
contenido_bitacora = """# Bitácora de Trabajo y Desarrollo

**Proyecto:** HRTech People Analytics — Equipo 8  
**Actividad:** Guía Práctica 04 — Gobernanza, Ética y Anonimización de Datos  
**Estado:** Completado / Listo para entrega  

---

## 1. Resumen de Actividades Realizadas

1. **Diseño del Marco de Gobernanza y Ética de Datos:**
   - **Clasificación de Privacidad:** Identificación y categorización de campos sensibles del dataset de personal (PII directos como `nombre_completo`, identificadores relacionales como `empleado_id`, datos financieros como `salario_mensual` y cuasi-identificadores como `edad`, `genero` y coordenadas geográficas).
   - **Definición de Políticas:** Elaboración del esquema de Control de Acceso basado en Roles (RBAC), reglas de Uso Aceptable (prohibición de reidentificación y decisiones automáticas) y políticas de retención/destrucción.
   - **Matriz de Riesgos:** Evaluación de riesgos éticos y de seguridad (reidentificación por cuasi-identificadores, sesgo algorítmico y fuga de PII en repositorios Git) con sus medidas de mitigación.

2. **Desarrollo del Cuaderno de Código (`guia_04_gobernanza_etica.ipynb`):**
   - Creación del cuaderno de Jupyter Notebook con celdas de documentación y código ejecutable en Python.
   - Implementación de la función de **Seudonimización mediante Hash SHA-256 con Salt** para los identificadores de empleados.
   - Eliminación directa de datos PII (`nombre_completo`).
   - Implementación de técnicas de **Generalización / Banding** para convertir salarios continuos y edades en rangos categóricos (`rango_salarial` y `rango_edad`).
   - Supresión de variables geográficas de alta precisión (`latitud`, `longitud`, direcciones exactas).
   - Generación automática y exportación del dataset procesado final (`dataset_anonimizado.csv`).

3. **Resolución de Incidencias Técnicas de Entorno:**
   - Diagnóstico y solución del error de Jupyter en VS Code (*"requires the ipykernel package"*).
   - Manejo de restricción de entorno administrado externamente por el gestor de paquetes `uv` (**PEP 668**).
   - Configuración e instalación exitosa de un entorno virtual aislado (`.venv`) con Python 3.12, `ipykernel`, `pandas` y `numpy`.

---

## 2. Registro de Incidencias y Soluciones Técnicas

| # | Incidencia / Error | Causa Raíz | Solución Aplicada |
| :-: | :--- | :--- | :--- |
| **1** | `requires the ipykernel package` en VS Code | El ejecutable de Python global gestionado por `uv` no tenía instalado el kernel de Jupyter. | Se creó un entorno virtual (`uv venv`) y se vinculó en el kernel de VS Code. |
| **2** | `externally-managed-environment` (PEP 668) | Protección de Python contra instalaciones globales con `pip`. | Se evitó la instalación global usando `uv venv --python 3.12` e instalando paquetes aislados mediante `uv pip install`. |
| **3** | Incompatibilidad potencial con Python 3.14 | Ciertas librerías de datos aún no tienen paquetes precompilados (*wheels*) para Python 3.14. | Se configuró el entorno virtual usando la versión estable **Python 3.12**. |

---

## 3. Entregables Generados

- **`guia_04_gobernanza_etica.ipynb`**: Cuaderno ejecutable con el pipeline completo de anonimización y generador de bitácora.
- **`bitacora_semana_04.md`**: Documento generado dinámicamente con la bitácora de desarrollo e incidencias.
- **`dataset_anonimizado.csv`**: Archivo de datos transformado listo para análisis seguro sin exposición de PII.
"""

# Guardar la bitácora en formato .md desde el notebook
ruta_bitacora = Path("../documentos/bitacora_semana_04.md")
with open(ruta_bitacora, "w", encoding="utf-8") as f:
    f.write(contenido_bitacora)

print(f"Archivo de bitácora generado exitosamente en: {ruta_bitacora.resolve()}")

Archivo de bitácora generado exitosamente en: D:\ITCA\CICLO 4 2026\GDIABD\Proyecto_Final\HRTechPeopleAnalytics_ProyectoFinal\documentos\bitacora_semana_04.md
